# Run RAG Query

This notebook is to run the RAG query. 

In [11]:
from pathlib import Path
from typing import List

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

# Load documents as PDFs
from langchain_community.document_loaders import PyPDFLoader

# Chunk documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_community.embeddings import HuggingFaceEmbeddings

# Create vector DB
from langchain_community.vectorstores import FAISS

print("Imports loaded.")

from rank_bm25 import BM25Okapi
from langchain_core.documents import Document
import re
import numpy as np

Imports loaded.


# Reload Vector DB 

In [12]:
# Embedding Model. Using free tier AI
# from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

/var/folders/r_/329g_gv939xfmdlhmvswc5bm0000gn/T/ipykernel_50480/1172382142.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10386.18it/s]


In [13]:
from langchain_community.vectorstores import FAISS

# Paths for each vector DB
fixed_vector_path = "fixed_rag_faiss_index"
structure_path = "faiss_structure_chunks"
sentence_path = "faiss_sentence_chunks"
semantic_path = "faiss_semantic_chunks"

# Load each vector DB

fixed_vector_db = FAISS.load_local(
    fixed_vector_path,
    embeddings,
    allow_dangerous_deserialization=True
)

structure_vector_db = FAISS.load_local(
    structure_path,
    embeddings,
    allow_dangerous_deserialization=True
)

sentence_vector_db = FAISS.load_local(
    sentence_path,
    embeddings,
    allow_dangerous_deserialization=True
)

semantic_vector_db = FAISS.load_local(
    semantic_path,
    embeddings,
    allow_dangerous_deserialization=True
)

print("Vector DB reloaded from local folder.")

Vector DB reloaded from local folder.


In [14]:
# Store all 4 vector DBs in one dictionary

vector_dbs = {
    "fixed_size": fixed_vector_db,
    "document_structure": structure_vector_db,
    "sentence": sentence_vector_db,
    "semantic": semantic_vector_db
}

print("Available vector DBs:", list(vector_dbs.keys()))

Available vector DBs: ['fixed_size', 'document_structure', 'sentence', 'semantic']


In [15]:
# Load local Hugging Face model for answer generation
# TinyLlama is a causal model, so we must format the prompt as chat and decode ONLY new tokens.

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Use CPU-safe loading. If you have a GPU, you can change this later.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map=None
)

model.eval()
print("LLM loaded.")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:04<00:00, 44.83it/s]


LLM loaded.


In [16]:
def llm(prompt: str, max_new_tokens: int = 350) -> str:
    """
    Generate an answer using TinyLlama.
    Important fix: decode only the newly generated tokens, not the full prompt.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful RAG assistant. Answer only from the provided context. "
                "Use citations exactly as provided, such as [1] or [1, p. 3]. "
                "If the context does not answer the question, say that the documents do not contain enough information."
                "Do not stop mid-sentence."
            )
        },
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # CRITICAL FIX: remove the prompt tokens from the output
    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()


In [17]:
# def retrieve_chunks(query: str, vector_db, k: int = 4):
#     """Retrieve top-k chunks from a selected FAISS vector database."""
#     return vector_db.similarity_search(query, k=k)


# # Quick retrieval test
# test_chunks = retrieve_chunks(
#     "What is copyright?",
#     vector_dbs["fixed_size"],
#     k=4
# )

# print(f"Retrieved {len(test_chunks)} chunks")

# for i, doc in enumerate(test_chunks, start=1):
#     print(f"\nChunk {i}")
#     print("Source:", doc.metadata.get("source_file", "Unknown file"))
#     print("Page:", doc.metadata.get("page", "Unknown page"))
#     print("Chunking:", doc.metadata.get("chunking_method", "fixed_size"))
#     print("Citation:", doc.metadata.get("ieee_citation", "Unknown citation"))
#     print("Preview:", doc.page_content[:250].replace("\n", " "))

In [18]:
# Different retrieval methods 

def retrieve_similarity(query: str, vector_db, k: int = 4):
    """Standard vector similarity search."""
    return vector_db.similarity_search(query, k=k)


def retrieve_mmr(query: str, vector_db, k: int = 4, fetch_k: int = 20):
    """
    MMR = Maximal Marginal Relevance.
    Good when you want diverse chunks instead of very similar repeated chunks.
    """
    return vector_db.max_marginal_relevance_search(
        query,
        k=k,
        fetch_k=fetch_k
    )


def retrieve_with_scores(query: str, vector_db, k: int = 4):
    """
    Vector search but keeps similarity scores in metadata.
    """
    results = vector_db.similarity_search_with_score(query, k=k)

    docs = []
    for doc, score in results:
        doc.metadata["similarity_score"] = float(score)
        docs.append(doc)

    return docs


def retrieve_threshold(query: str, vector_db, k: int = 8, max_score: float = 1.0):
    """
    Keeps only chunks under a distance threshold.
    For FAISS, lower score usually means more similar.
    Adjust max_score after inspecting your scores.
    """
    results = vector_db.similarity_search_with_score(query, k=k)

    filtered_docs = []
    for doc, score in results:
        if score <= max_score:
            doc.metadata["similarity_score"] = float(score)
            filtered_docs.append(doc)

    return filtered_docs

In [19]:
# BM25 keyword retrieval 

def tokenize(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


def build_bm25_index(vector_db):
    """
    Extracts documents from a FAISS vector DB and builds a BM25 keyword index.
    """
    docs = list(vector_db.docstore._dict.values())
    tokenized_docs = [tokenize(doc.page_content) for doc in docs]
    bm25 = BM25Okapi(tokenized_docs)

    return docs, bm25


bm25_indexes = {}

for method_name, db in vector_dbs.items():
    docs, bm25 = build_bm25_index(db)
    bm25_indexes[method_name] = {
        "docs": docs,
        "bm25": bm25
    }

print("BM25 indexes built for:", list(bm25_indexes.keys()))

BM25 indexes built for: ['fixed_size', 'document_structure', 'sentence', 'semantic']


In [20]:
def retrieve_bm25(query: str, method_name: str, k: int = 4):
    """
    Keyword-based retrieval.
    Good when the question uses exact terms from the papers.
    """
    docs = bm25_indexes[method_name]["docs"]
    bm25 = bm25_indexes[method_name]["bm25"]

    tokenized_query = tokenize(query)
    scores = bm25.get_scores(tokenized_query)

    top_indices = np.argsort(scores)[::-1][:k]

    retrieved_docs = []
    for idx in top_indices:
        doc = docs[idx]
        doc.metadata["bm25_score"] = float(scores[idx])
        retrieved_docs.append(doc)

    return retrieved_docs

In [21]:
# hybrid retrieval: BM25 + vector search 

def retrieve_hybrid(query: str, vector_db, method_name: str, k: int = 4, vector_k: int = 8, bm25_k: int = 8):
    """
    Hybrid retrieval combines:
    1. semantic vector search
    2. keyword BM25 search

    This is usually stronger than either one alone.
    """
    vector_docs = retrieve_similarity(query, vector_db, k=vector_k)
    bm25_docs = retrieve_bm25(query, method_name, k=bm25_k)

    combined = []
    seen = set()

    for doc in vector_docs + bm25_docs:
        key = doc.page_content[:200]

        if key not in seen:
            combined.append(doc)
            seen.add(key)

        if len(combined) >= k:
            break

    return combined

In [22]:
# unified retrieval method 
def retrieve_chunks(
    query: str,
    vector_db,
    method_name: str,
    retrieval_method: str = "similarity",
    k: int = 4
):
    """
    Select retrieval method.
    """

    if retrieval_method == "similarity":
        return retrieve_similarity(query, vector_db, k=k)

    elif retrieval_method == "mmr":
        return retrieve_mmr(query, vector_db, k=k)

    elif retrieval_method == "with_scores":
        return retrieve_with_scores(query, vector_db, k=k)

    elif retrieval_method == "threshold":
        return retrieve_threshold(query, vector_db, k=k)

    elif retrieval_method == "bm25":
        return retrieve_bm25(query, method_name, k=k)

    elif retrieval_method == "hybrid":
        return retrieve_hybrid(query, vector_db, method_name, k=k)

    else:
        raise ValueError(f"Unknown retrieval method: {retrieval_method}")

In [34]:
def rag_query(question: str, vector_db, method_name: str, retrieval_method: str = "similarity", k: int = 5) -> dict:
    """
    RAG pipeline for one chunking method.
    """
    try:
        retrieved_docs = retrieve_chunks(
            question,
            vector_db,
            method_name=method_name,
            retrieval_method=retrieval_method,
            k=k
        )

        if not retrieved_docs:
            return {
                "method": method_name,
                "question": question,
                "answer": "I could not retrieve any relevant document chunks.",
                "retrieved_docs": []
            }

        context_blocks = []

        for i, doc in enumerate(retrieved_docs, start=1):
            page = doc.metadata.get("page", "Unknown page")
            source_file = doc.metadata.get("source_file", "Unknown file")
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            context_blocks.append(
                f"""
[{i}]
File: {source_file}
Page: {page}
Citation: {citation}

Text:
{doc.page_content[:700]}
"""
            )

        context = "\n\n---\n\n".join(context_blocks)

        prompt = f"""
Answer the question using only the sources below.
Use citations like [1] or [2].
Keep the answer concise.
Do not stop mid-sentence.

Question:
{question}

Sources:
{context}

Answer:
"""

        answer = llm(prompt)

        return {
            "method": method_name,
            "question": question,
            "answer": answer,
            "retrieved_docs": retrieved_docs,
            "retrieval_method": retrieval_method
        }

    except Exception as e:
        return {
            "method": method_name,
            "question": question,
            "error": str(e)
        }

In [24]:
# Test one method

def print_rag_query_answer(question, vdb, method_name, k=2): 
    result = rag_query(
        question,
        vdb,
        method_name,
        k=2
    )

    if "error" in result:
        print("RAG failed:", result["error"])
    else:
        print("Method:", result["method"])
        print("Question:", result["question"])
        print("\nAnswer:\n")
        print(result["answer"])

        print("\nREFERENCES:\n")

        seen = set()

        for i, doc in enumerate(result["retrieved_docs"], start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            # Avoid duplicate citations
            if citation not in seen:
                print(f"[{i}] {citation}")
                seen.add(citation)
    return 

# print_rag_query_answer(
#     "What is the impact of LLMs on copyright?",
#     vector_dbs["fixed_size"],
#     method_name="Fixed Size",
#     k=2
# )

In [25]:
import os

def compare_chunking(query, output_file, k=2):
    output = []

    for method_name, db in vector_dbs.items():
        output.append("\n" + "=" * 80)
        output.append(f"METHOD: {method_name}")
        output.append("=" * 80)

        result = rag_query(
            query,
            db,
            method_name=method_name,
            k=k
        )

        if "error" in result:
            output.append(f"RAG failed: {result['error']}")
            continue

        output.append(f"Question: {result['question']}")
        output.append("\nAnswer:\n")
        output.append(result["answer"])

        output.append("\nREFERENCES:\n")

        seen = set()

        for i, doc in enumerate(result["retrieved_docs"], start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")

            if citation not in seen:
                output.append(f"[{i}] {citation}")
                seen.add(citation)

        output.append("\nRETRIEVED CHUNKS:")

        for i, doc in enumerate(result["retrieved_docs"], start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")
            page = doc.metadata.get("page", "Unknown")
            chunking = doc.metadata.get("chunking_method", method_name)
            preview = doc.page_content[:300].replace("\n", " ")

            output.append(f"\nChunk {i}")
            output.append(f"Source: {citation}")
            output.append(f"Page: {page}")
            output.append(f"Chunking: {chunking}")
            output.append(f"Preview: {preview}")

    # Ensure directory exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(output))

    print(f"Results saved to {output_file}")

In [26]:
# q = "What is the impact of LLMs on copyright?"
# ofile = "outputs/rag_comparison_test_1.txt"
# compare_chunking(q, ofile, k=2)

In [27]:
# q = "What do some existing papers say on copyright?"
# ofile = "outputs/copyright.txt"
# compare_chunking(q, ofile, k=5)

In [28]:
# q = "What are some policy implications of recent tech?"
# ofile = "outputs/policy_implication.txt"
# compare_chunking(q, ofile, k=5)

Personal preference for best chunking method is document structure. 

# Compare retrieval methods

In [29]:
retrieval_methods = [
    "similarity",
    "mmr",
    "with_scores",
    "bm25",
    "hybrid"
]

In [ ]:
def compare_retrieval_methods(query, output_file, chunking_method="fixed_size", k=4):
    output = []

    vector_db = vector_dbs[chunking_method]

    for retrieval_method in retrieval_methods:
        output.append("\n" + "=" * 80)
        output.append(f"CHUNKING METHOD: {chunking_method}")
        output.append(f"RETRIEVAL METHOD: {retrieval_method}")
        output.append("=" * 80)

        result = rag_query(
            query,
            vector_db,
            method_name=chunking_method,
            retrieval_method=retrieval_method,
            k=k
        )

        if "error" in result:
            output.append(f"RAG failed: {result['error']}")
            continue

        output.append(f"Question: {result['question']}")
        output.append("\nAnswer:\n")
        output.append(result["answer"])

        output.append("\nRETRIEVED CHUNKS:")

        for i, doc in enumerate(result["retrieved_docs"], start=1):
            citation = doc.metadata.get("ieee_citation", "Unknown citation")
            page = doc.metadata.get("page", "Unknown")
            chunking = doc.metadata.get("chunking_method", chunking_method)
            preview = doc.page_content[:300].replace("\n", " ")

            output.append(f"\nChunk {i}")
            output.append(f"Source: {citation}")
            output.append(f"Page: {page}")
            output.append(f"Chunking: {chunking}")

            if "similarity_score" in doc.metadata:
                output.append(f"Similarity score: {doc.metadata['similarity_score']}")

            if "bm25_score" in doc.metadata:
                output.append(f"BM25 score: {doc.metadata['bm25_score']}")

            output.append(f"Preview: {preview}")

    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        f.write("\n".join(output))

    print(f"Results saved to {output_file}")

In [35]:
q = "What is the impact of LLMs on copyright?"

compare_retrieval_methods(
    q,
    "outputs/retrieval_method_comparison.txt",
    chunking_method="document_structure",
    k=4
)

[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=350) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Results saved to outputs/retrieval_method_comparison.txt
